In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

# 1. すでに作成した4番タイプ表を読み込む
type_df = pd.read_csv("npb_2023_main_fourth_batter_types.csv")

# 2. NPB公式の2023年チーム打撃成績URL
CENTRAL_URL = "https://npb.jp/bis/2023/stats/tmb_c.html"
PACIFIC_URL = "https://npb.jp/bis/2023/stats/tmb_p.html"

def load_team_runs(url: str) -> pd.DataFrame:
    res = requests.get(url, timeout=20)
    res.raise_for_status()
    res.encoding = res.apparent_encoding

    soup = BeautifulSoup(res.text, "html.parser")
    text = soup.get_text("\n", strip=True)

    rows = []

    if "tmb_c" in url:
        # セ・リーグ
        team_pattern = r"(DeNA|巨\s*人|中\s*日|ヤクルト|阪\s*神|広\s*島)"
    else:
        # パ・リーグ
        team_pattern = r"(ソフトバンク|日本ハム|ロッテ|楽\s*天|オリックス|西\s*武)"

    # 行の並び:
    # チーム名 打率 試合 打席 打数 得点 ...
    pattern = re.compile(
        rf"{team_pattern}\s*\.?\d+\s+(\d+)\s+\d+\s+\d+\s+(\d+)"
    )

    for m in pattern.finditer(text):
        team = re.sub(r"\s+", "", m.group(1))
        games = int(m.group(2))
        runs = int(m.group(3))
        rows.append([team, games, runs])

    if not rows:
        raise ValueError(f"チーム成績を抽出できませんでした: {url}")

    df = pd.DataFrame(rows, columns=["チーム", "試合", "得点"])
    return df

# 3. セ・パを読み込んで結合
central_df = load_team_runs(CENTRAL_URL)
pacific_df = load_team_runs(PACIFIC_URL)
team_runs_df = pd.concat([central_df, pacific_df], ignore_index=True)

# 4. 球団名をあなたのCSV側に合わせる
name_map = {
    "DeNA": "DeNA",
    "ソフトバンク": "ソフトバンク",
    "日本ハム": "日本ハム",
    "オリックス": "オリックス",
    "ロッテ": "ロッテ",
    "楽天": "楽天",
    "西武": "西武",
    "巨人": "巨人",
    "阪神": "阪神",
    "広島": "広島",
    "ヤクルト": "ヤクルト",
    "中日": "中日",
}

team_runs_df["球団"] = team_runs_df["チーム"].replace(name_map)

# 5. 数値化
team_runs_df["試合"] = pd.to_numeric(team_runs_df["試合"], errors="coerce")
team_runs_df["得点"] = pd.to_numeric(team_runs_df["得点"], errors="coerce")

# 6. 1試合平均得点を作成
team_runs_df["1試合平均得点"] = team_runs_df["得点"] / team_runs_df["試合"]

# 7. 4番タイプ表と結合
merged_df = type_df.merge(
    team_runs_df[["球団", "試合", "得点", "1試合平均得点"]],
    on="球団",
    how="left"
)

# 8. 保存
merged_df.to_csv(
    "npb_2023_main_fourth_batter_types_with_team_runs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=== チーム得点データ ===")
print(team_runs_df)

print("\n=== 4番タイプ × チーム得点 ===")
print(merged_df)

print("\n=== タイプ別平均 ===")
summary = (
    merged_df.groupby("4番タイプ")[["得点", "1試合平均得点"]]
    .mean()
    .round(3)
    .reset_index()
)
print(summary)

print("\n保存完了: npb_2023_main_fourth_batter_types_with_team_runs.csv")

=== チーム得点データ ===
       チーム   試合   得点      球団   1試合平均得点
0       巨人  143  523      巨人  3.657343
1     DeNA  143  520    DeNA  3.636364
2       阪神  143  555      阪神  3.881119
3       広島  143  493      広島  3.447552
4     ヤクルト  143  534    ヤクルト  3.734266
5       中日  143  390      中日  2.727273
6    オリックス  143  508   オリックス  3.552448
7   ソフトバンク  143  536  ソフトバンク  3.748252
8       楽天  143  513      楽天  3.587413
9      ロッテ  143  505     ロッテ  3.531469
10      西武  143  435      西武  3.041958
11    日本ハム  143  464    日本ハム  3.244755

=== 4番タイプ × チーム得点 ===
        球団     選手名  4番出場回数  本塁打    OBP    SLG    OPS   打点  長打型スコア  総合型スコア  \
0     DeNA    牧 秀悟     143   29  0.337  0.530  0.867  103   0.781   0.341   
1    オリックス    森 友哉      71   18  0.385  0.508  0.893   64   0.136   0.839   
2   ソフトバンク   柳田 悠岐      74   22  0.378  0.484  0.861   85   0.168   0.544   
3     ヤクルト   村上 宗隆     139   31  0.375  0.500  0.875   84   0.764   0.634   
4      ロッテ    ポランコ      88   26  0.312  0.450  0.762   75   0.006  -